# 7장 웹부록 | 컴퓨터로 여는 액설로드 대회

「영화로 보는 AI 리터러시」 7장 '전략적으로 생각하는 AI: 게임이론과 강화학습'의 코랩 실습입니다.

**첫 번째 부분**에서는 팃포탯, 늘 배신, 늘 협력, 무작위 같은 전략을 코드로 만들어 반복 죄수의 딜레마 리그전을 엽니다. 여러분이 직접 새 전략을 한 줄로 만들어 넣고 순위를 확인합니다.
**두 번째 부분**에서는 작은 미로에서 Q-러닝 로봇이 보물을 찾아가는 과정을 칸마다 색으로 봅니다.

**사용법**
- 위에서부터 차례로 각 칸 왼쪽의 ▶(실행) 버튼을 누르세요. 코드를 몰라도 괜찮습니다.
- `# ← 바꿔 보세요`라고 적힌 곳만 고쳐서 다시 ▶를 누르면 됩니다.
- 처음부터 다시 하고 싶으면 메뉴에서 **런타임 → 모두 실행**을 누르세요.
- 그래프 글자는 한글이 깨지지 않도록 영어로 적었습니다.

# 첫 번째 부분: 액설로드 대회

## 1단계. 점수 규칙과 전략 만들기

실습 1과 같은 점수표(보수표)입니다.

| | 상대 협력 | 상대 배신 |
|---|---|---|
| **나 협력** | 3점 | 0점 |
| **나 배신** | 5점 | 1점 |

전략은 '지금까지 내가 낸 것(`me`)과 상대가 낸 것(`them`)을 보고 이번에 무엇을 낼지' 정하는 규칙입니다. `"C"`는 협력, `"D"`는 배신입니다.
`them[-1]`은 '상대가 바로 지난번에 낸 것'이라는 뜻입니다.

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PAYOFF = {("C", "C"): (3, 3), ("C", "D"): (0, 5), ("D", "C"): (5, 0), ("D", "D"): (1, 1)}

def tester(me, them):
    """첫판에 배신해 떠보고, 상대가 갚지 않으면 이용하고, 갚으면 사과한 뒤 팃포탯처럼 합니다."""
    if not me:
        return "D"                                   # 첫판: 떠보기
    if "D" in them[1:]:                              # 상대가 갚았다면
        if them[1:].count("D") == 1 and them[-1] == "D":
            return "C"                               # 한 번 사과
        return them[-1]                              # 그다음부터는 팃포탯
    return "D" if len(me) % 2 == 0 else "C"          # 갚지 않는 상대는 번갈아 이용

STRATEGIES = {
    # 이름: 규칙 (me = 내 기록, them = 상대 기록)
    "TitForTat (팃포탯)":       lambda me, them: "C" if not them else them[-1],
    "AlwaysDefect (늘 배신)":   lambda me, them: "D",
    "AlwaysCooperate (늘 협력)": lambda me, them: "C",
    "Random (무작위)":          lambda me, them: random.choice("CD"),
    "Grudger (원한)":           lambda me, them: "D" if "D" in them else "C",
    "Joss (몰래 배신)":         lambda me, them: "D" if (them and them[-1] == "D") or random.random() < 0.1 else "C",
    "SuspiciousTFT (의심 팃포탯)": lambda me, them: "D" if not them else them[-1],
    "Tester (떠보기)":          lambda me, them: tester(me, them),
}
for name in STRATEGIES:
    print("-", name)

- **원한**: 처음엔 협력하다가, 상대가 한 번이라도 배신하면 끝까지 배신합니다.
- **몰래 배신**: 팃포탯처럼 하다가 열 번에 한 번쯤 몰래 배신합니다.
- **의심 팃포탯**: 팃포탯과 같지만 첫판을 배신으로 시작합니다.
- **떠보기**: 첫판에 배신해 상대를 떠봅니다. 상대가 갚지 않으면 번갈아 배신하며 이용하고, 갚으면 한 번 사과(협력)한 뒤 팃포탯처럼 합니다.

## 2단계. 한 경기 해 보기

두 전략이 200판을 겨룹니다(액설로드 대회와 같은 판 수). 처음 10판의 기록과 최종 점수를 봅니다.

In [ ]:
ROUNDS = 200

def play(name_a, name_b, rounds=ROUNDS, seed=0, noise=0.0):
    random.seed(seed)
    a, b = STRATEGIES[name_a], STRATEGIES[name_b]
    hist_a, hist_b, score_a, score_b = [], [], 0, 0
    for _ in range(rounds):
        x, y = a(hist_a, hist_b), b(hist_b, hist_a)
        if random.random() < noise: x = "C" if x == "D" else "D"   # 실수로 반대 카드를 내는 경우
        if random.random() < noise: y = "C" if y == "D" else "D"
        pa, pb = PAYOFF[(x, y)]
        score_a, score_b = score_a + pa, score_b + pb
        hist_a.append(x); hist_b.append(y)
    return score_a, score_b, hist_a, hist_b

A, B = "TitForTat (팃포탯)", "Joss (몰래 배신)"   # ← 다른 두 전략 이름을 넣어 보세요
sa, sb, ha, hb = play(A, B)
print(f"{A:28s}: {' '.join(ha[:10])} ...")
print(f"{B:28s}: {' '.join(hb[:10])} ...")
print(f"\n200판 합계  {A}: {sa}점,  {B}: {sb}점")

## 3단계. 나만의 전략 한 줄로 만들기

👉 **바꿔 볼 곳**: 아래 `MY_STRATEGY` 한 줄을 고쳐 나만의 전략을 만드세요. 예시를 참고하세요.

```python
lambda me, them: "C"                                         # 늘 협력
lambda me, them: "C" if len(them) < 2 else them[-2]          # 두 판 전의 상대를 따라 하기
lambda me, them: "D" if them.count("D") > len(them) / 3 else "C"   # 상대 배신이 1/3을 넘으면 배신
lambda me, them: "D" if len(me) % 5 == 4 else "C"            # 다섯 판마다 한 번 배신
```

In [ ]:
MY_STRATEGY = lambda me, them: "C" if len(them) < 2 else them[-2]   # ← 이 한 줄을 바꿔 보세요

STRATEGIES["Mine (내 전략)"] = MY_STRATEGY
print("내 전략을 리그에 넣었습니다. 참가 전략:", len(STRATEGIES), "개")

## 4단계. 리그전 열기

모든 전략이 서로 한 번씩(자기 자신과도) 200판을 겨룹니다. 점수를 모두 더해 순위를 매깁니다.
'이긴 경기 수'는 한 경기에서 상대보다 점수가 높았던 횟수입니다.

👉 **바꿔 볼 곳**: `NOISE`는 실수로 반대 카드를 낼 확률입니다. `0.05`(5%)로 바꿔 보세요.

In [ ]:
NOISE = 0.0   # ← 실수 확률 (0.05 등으로 바꿔 보세요)

names = list(STRATEGIES)
table = np.zeros((len(names), len(names)))
wins = {n: 0 for n in names}
for i, a in enumerate(names):
    for j, b in enumerate(names):
        if j < i:
            continue
        sa, sb, _, _ = play(a, b, seed=1000 * i + j, noise=NOISE)
        table[i, j], table[j, i] = sa, sb
        if a != b:
            if sa > sb: wins[a] += 1
            elif sb > sa: wins[b] += 1

result = pd.DataFrame({"총점": table.sum(axis=1).astype(int),
                       "경기당 평균": (table.sum(axis=1) / len(names)).round(0).astype(int),
                       "이긴 경기 수": [wins[n] for n in names]}, index=names)
result = result.sort_values("총점", ascending=False)
result.index.name = "전략"
result.insert(0, "순위", range(1, len(result) + 1))
result

In [ ]:
plt.figure(figsize=(8, 4))
order = result.index[::-1]
colors = ["#e8743b" if "Mine" in n else ("#2ca02c" if "TitForTat" in n else "#5b8def") for n in order]
plt.barh([n.split(" (")[0] for n in order], result.loc[order, "총점"], color=colors)
plt.xlabel("Total score")
plt.title(f"Round-robin tournament (noise = {NOISE})")
plt.tight_layout()
plt.show()

tft = "TitForTat (팃포탯)"
print(f"팃포탯이 이긴 경기 수: {wins[tft]}  (책: '팃포탯은 어떤 상대도 한 판에서 이기지 못합니다')")

팃포탯은 한 경기도 이기지 못합니다. 기껏해야 비깁니다. 그래도 총점은 높은 편입니다.

다만 이렇게 작은 리그에서는 **누가 참가했는지**에 따라 순위가 크게 바뀝니다. 착한 전략만 많으면 '늘 협력'도 잘 해냅니다. 액설로드의 대회에는 훨씬 다양한 전략이 모였습니다.

## 5단계. 더 해 보기: 살아남기 게임

실습 1의 '신뢰의 진화'처럼, 점수를 많이 얻은 전략은 다음 세대에 더 많아지고 점수가 낮은 전략은 줄어든다고 해 봅니다(액설로드도 이 실험을 했습니다).
처음에는 모든 전략이 같은 수로 시작합니다. 세대가 지나면 누가 남을까요?

In [ ]:
GENERATIONS = 30   # ← 세대 수

share = np.ones(len(names)) / len(names)
history = [share.copy()]
for g in range(GENERATIONS):
    fitness = table @ share            # 지금 마을 구성에서 각 전략이 얻는 평균 점수
    share = share * fitness
    share = share / share.sum()        # 점수에 비례해 다음 세대의 비율이 정해집니다
    history.append(share.copy())
history = np.array(history)

plt.figure(figsize=(9, 4.5))
for k, n in enumerate(names):
    plt.plot(history[:, k], label=n.split(" (")[0], lw=3 if "TitForTat" in n else 1.5)
plt.xlabel("Generation")
plt.ylabel("Share of population")
plt.title("Who survives?")
plt.legend(bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
plt.show()
for n, s in sorted(zip(names, share), key=lambda x: -x[1]):
    print(f"{n:28s} {s:.0%}")

처음에는 '늘 배신'이 '늘 협력'을 이용해 늘어납니다. 그런데 먹잇감이 사라지면 '늘 배신'도 줄어듭니다. 끝까지 남는 것은 먼저 배신하지 않되, 배신에는 갚는 전략들입니다.

# 두 번째 부분: Q-러닝 로봇의 미로 찾기

## 6단계. 미로 만들기

로봇(에이전트)은 `S`에서 출발해 보물 `G`를 찾아야 합니다. `#`은 벽, `X`는 함정입니다.

- 한 칸 움직일 때마다 보상 −1 (빨리 찾을수록 좋습니다)
- 보물에 닿으면 보상 +10, 함정에 빠지면 보상 −10, 그리고 한 판(에피소드)이 끝납니다.

로봇은 미로 지도를 모릅니다. 해 보고 받은 보상만으로 배웁니다(3-5절).

👉 **바꿔 볼 곳**: `MAZE`의 글자를 바꿔 나만의 미로를 만들어 보세요. 줄마다 글자 수가 같아야 합니다.

In [ ]:
MAZE = [
    "S..#....",
    ".#.#.##.",
    ".#...#..",
    ".####.#.",
    "......#X",
    ".##.#...",
    "...#..#G",
]   # ← 미로를 바꿔 보세요 (S 출발, G 보물, # 벽, X 함정)

H, W = len(MAZE), len(MAZE[0])
start = next((r, c) for r in range(H) for c in range(W) if MAZE[r][c] == "S")
ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]          # 위, 아래, 왼쪽, 오른쪽
ARROWS = ["↑", "↓", "←", "→"]

def move(state, a):
    r, c = state[0] + ACTIONS[a][0], state[1] + ACTIONS[a][1]
    if not (0 <= r < H and 0 <= c < W) or MAZE[r][c] == "#":
        r, c = state                                     # 벽이면 제자리
    cell = MAZE[r][c]
    if cell == "G":
        return (r, c), 10, True
    if cell == "X":
        return (r, c), -10, True
    return (r, c), -1, False

print("\n".join(" ".join(row) for row in MAZE))

## 7단계. Q-러닝으로 배우기

Q-러닝은 '칸마다, 행동마다 점수(Q값)를 적은 표'를 만듭니다(3-5절 심화 상식).
행동을 하고 보상을 받을 때마다 표의 점수를 **'방금 받은 보상 + 다음 칸에서 기대할 수 있는 가장 좋은 점수'** 쪽으로 조금 옮깁니다.

- `EPSILON`: 탐험 비율입니다. 0.1이면 열 번에 한 번은 무작위로 움직입니다(ε-탐욕).
- `ALPHA`: 한 번에 점수를 얼마나 고칠지(학습률)
- `GAMMA`: 나중 보상을 얼마나 중요하게 볼지

👉 **바꿔 볼 곳**: `EPSILON`을 0이나 0.5로, `EPISODES`를 줄이거나 늘려 보세요.

In [ ]:
EPISODES = 300   # ← 몇 판 연습할지
EPSILON = 0.1    # ← 탐험 비율
ALPHA = 0.5      # ← 학습률
GAMMA = 0.95     # ← 나중 보상의 중요도
SNAPSHOTS = [1, 10, 50, EPISODES]   # 이 판들이 끝났을 때의 Q표를 저장합니다

rng = random.Random(0)
Q = np.zeros((H, W, 4))
steps_per_episode, saved = [], {}
for ep in range(1, EPISODES + 1):
    state, done, steps = start, False, 0
    while not done and steps < 200:
        if rng.random() < EPSILON:
            a = rng.randrange(4)                                  # 탐험: 아무 쪽으로나
        else:
            best = Q[state].max()
            a = rng.choice([k for k in range(4) if Q[state][k] == best])   # 활용: 가장 좋은 쪽으로
        nxt, reward, done = move(state, a)
        target = reward if done else reward + GAMMA * Q[nxt].max()
        Q[state][a] += ALPHA * (target - Q[state][a])             # 점수표 고치기
        state, steps = nxt, steps + 1
    steps_per_episode.append(steps)
    if ep in SNAPSHOTS:
        saved[ep] = Q.copy()

plt.figure(figsize=(8, 3.5))
plt.plot(steps_per_episode)
plt.xlabel("Episode")
plt.ylabel("Steps to finish")
plt.title("The robot finds the treasure faster and faster")
plt.show()
print(f"처음 10판 평균 {np.mean(steps_per_episode[:10]):.0f}걸음 → 마지막 10판 평균 {np.mean(steps_per_episode[-10:]):.0f}걸음")

## 8단계. 칸마다 색으로 보기

각 칸의 색은 '이 칸에서 앞으로 받을 수 있는 가장 좋은 점수'(가치)입니다. 노란색일수록 높고, 보라색일수록 낮습니다. 화살표는 그 칸에서 로봇이 고르는 행동입니다.
처음에는 모든 칸의 점수가 0에서 시작합니다. 연습이 쌓일수록 보물 쪽에서부터 밝은 색이 번져 나오는 모습을 보세요. 3-5절의 '늦게 오는 보상을 거꾸로 나누기'(그림 7-10)입니다.

In [ ]:
def draw_q(Qs, ax, title, vmin=None, vmax=None):
    value = Qs.max(axis=2)
    grid = np.full((H, W), np.nan)
    for r in range(H):
        for c in range(W):
            if MAZE[r][c] not in "#GX":
                grid[r, c] = value[r, c]
    ax.imshow(grid, cmap="viridis", vmin=vmin, vmax=vmax)
    for r in range(H):
        for c in range(W):
            ch = MAZE[r][c]
            if ch == "#":
                ax.add_patch(plt.Rectangle((c - .5, r - .5), 1, 1, color="black"))
            elif ch in "GX":
                ax.add_patch(plt.Rectangle((c - .5, r - .5), 1, 1, color="gold" if ch == "G" else "crimson"))
                ax.text(c, r, ch, ha="center", va="center", fontsize=12, color="black" if ch == "G" else "white")
            elif Qs[r, c].any():
                ax.text(c, r, ARROWS[int(Qs[r, c].argmax())], ha="center", va="center", color="white", fontsize=12)
    ax.text(start[1], start[0] + 0.35, "S", ha="center", va="center", color="red", fontsize=9)
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])

fig, axes = plt.subplots(1, len(SNAPSHOTS), figsize=(4 * len(SNAPSHOTS), 4))
final_values = Q.max(axis=2)[[[MAZE[r][c] not in "#GX" for c in range(W)] for r in range(H)]]
vmin, vmax = final_values.min(), final_values.max()      # 모든 그림에 같은 색 기준을 씁니다
for ax, ep in zip(axes, SNAPSHOTS):
    draw_q(saved[ep], ax, f"after {ep} episode(s)", vmin, vmax)
plt.tight_layout()
plt.show()

마지막으로, 다 배운 로봇이 탐험 없이 가장 좋은 쪽으로만 움직이면 어떤 길로 가는지 봅니다.

In [ ]:
state, path = start, [start]
for _ in range(60):
    state, reward, done = move(state, int(Q[state].argmax()))
    path.append(state)
    if done:
        break
picture = [list(row) for row in MAZE]
for (r, c) in path[1:-1]:
    picture[r][c] = "*"
print("\n".join(" ".join(row) for row in picture))
print(f"\n{len(path) - 1}걸음 만에 {'보물에 도착했습니다!' if MAZE[path[-1][0]][path[-1][1]] == 'G' else '보물에 닿지 못했습니다.'}")

## 생각해 볼 것

1. 팃포탯은 한 경기도 이기지 못했는데 총점이나 살아남기 게임에서는 강했습니다. 이 결과가 학교나 조별 과제에 주는 교훈은 무엇일까요?
2. `EPSILON`을 0으로 하면 로봇은 탐험을 하지 않습니다. 결과가 어떻게 달라졌습니까? '단골집이냐 새 식당이냐'(탐험과 활용)로 설명해 보세요.
3. 함정의 보상을 −10에서 +20으로 바꾸면 로봇은 무엇을 배울까요? 3-8절의 보상 해킹(보트 경주 AI)과 비교해 보세요.